<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" width="400">

# Python Pickle MySQL Data & Pandas DataFrame Serialization Practice in Google Colab

**Main Tutorial Link:** [Python Pickle MySQL Tutorial](https://www.plus2net.com/python/pickle-mysql.php)

---

## 1. Environment Setup & Local MySQL/MariaDB Server Daemon

Install MariaDB server and initialize the local daemon within the Google Colab environment.

In [ ]:
# Stop and clean up any previously running MariaDB/MySQL instances completely
!pkill -9 mariadbd || true
!pkill -9 mysqld || true
!pkill -9 mysqld_safe || true
!rm -rf /var/run/mysqld/* /var/lib/mysql/* /run/mysqld/* || true

# Install MariaDB server and pymysql/pandas drivers
!apt-get update && apt-get install -y mariadb-server > /dev/null
!pip install pymysql cryptography pandas sqlalchemy > /dev/null

# Ensure run directories exist with proper system permissions
!mkdir -p /run/mysqld /var/run/mysqld /var/lib/mysql
!chown -R mysql:mysql /run/mysqld /var/run/mysqld /var/lib/mysql
!chmod 777 /run/mysqld /var/run/mysqld

# Initialize default database schema
!mysql_install_db --user=mysql --basedir=/usr --datadir=/var/lib/mysql > /dev/null

# Start MariaDB daemon in the background
import subprocess
import time
import os

print("Starting MariaDB daemon process...")
subprocess.Popen([
    "/usr/sbin/mariadbd",
    "--user=mysql",
    "--pid-file=/run/mysqld/mysqld.pid",
    "--socket=/run/mysqld/mysqld.sock",
    "--datadir=/var/lib/mysql",
    "--skip-networking=0"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Dynamically wait and poll for the socket file to become active
success = False
for i in range(15):
    if os.path.exists("/run/mysqld/mysqld.sock"):
        success = True
        print(f"Database socket found active after {i} seconds!")
        break
    time.sleep(1)

if not success:
    print("Warning: Database socket not found yet. Attempting setup anyway...")

# Create database and configure dedicated credentials via administrative socket connection
!mysql --socket=/run/mysqld/mysqld.sock -u root -e "CREATE DATABASE IF NOT EXISTS my_tutorial; \
  CREATE USER IF NOT EXISTS 'demo_user'@'localhost' IDENTIFIED BY 'demo_password'; \
  GRANT ALL PRIVILEGES ON my_tutorial.* TO 'demo_user'@'localhost'; \
  FLUSH PRIVILEGES;"

print("MySQL/MariaDB server running and user 'demo_user' configured successfully!")

## 2. Download SQL Dump (`student.sql`) & Import Database

Fetch `student.sql` directly from plus2net and import it into the local MySQL database.

In [ ]:
# Download the student.sql dump file into the Colab environment
!wget -q -O student.sql https://www.plus2net.com/python/download/student.sql || echo "Note: Ensure student.sql is hosted at this path"

# Import student.sql into the my_tutorial database
!mysql --socket=/run/mysqld/mysqld.sock -u root my_tutorial < student.sql

print("Database table 'student' created and populated successfully via student.sql!")

## 3. Pickling MySQL Data Using Pandas & Pickle Module

Read SQL data into a DataFrame and pickle it using both standard `pickle` and Pandas `to_pickle()`.

In [ ]:
import pickle
import pymysql
import pandas as pd

# Connect using PyMySQL
my_connect = pymysql.connect(
    host='localhost',
    user='demo_user',
    password='demo_password',
    database='my_tutorial',
    unix_socket='/run/mysqld/mysqld.sock'
)

# Read SQL into DataFrame
my_data = pd.read_sql("SELECT * FROM student LIMIT 0, 5", my_connect)
my_connect.close()

print("DataFrame loaded from MySQL:")
print(my_data)

# Method 1: Using standard pickle.dump()
with open('my_student', 'wb') as fob:
    pickle.dump(my_data, fob)

print("Pickled successfully using pickle.dump()!")

## 4. Un-Pickling & Using Pandas to_pickle() / read_pickle()

Demonstrate reading back pickled data and utilizing Pandas built-in serialization methods.

In [ ]:
# Read back using standard pickle.load()
with open('my_student', 'rb') as fob:
    restored_dict = pickle.load(fob)

print("Restored via pickle.load():")
print(restored_dict)

# Using Pandas to_pickle()
my_data.to_pickle("my_student_pandas")

# Using Pandas read_pickle()
restored_pandas = pd.read_pickle("my_student_pandas")
print("Restored via pd.read_pickle():")
print(restored_pandas)

---
### Visit [www.plus2net.com](https://www.plus2net.com) for more Python tutorials, code snippets, and data science guides.